In [ ]:

import pandas as pd
from sqlalchemy import create_engine


df = pd.read_csv("nyc_taxi_data.csv", parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
df = df.dropna().copy()#remove null and missing values


#positive numbers only
df = df[(df["passenger_count"] > 0)
    & (df["trip_distance"] > 0)
    & (df["fare_amount"] > 0)
    & (df["total_amount"] > 0)]


#numbers greater or equal to zero
non_negative_cols = ["extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "congestion_surcharge"]
for col in non_negative_cols:
    if col in df.columns:
        df = df[df[col] >= 0]

#dropoff must end before a pickup
df = df[df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]]


#check values within valid range
valid_ratecodes = [1, 2, 3, 4, 5, 6]
valid_payment_types = [1, 2, 3, 4, 5, 6]
df = df[df["RatecodeID"].isin(valid_ratecodes)
    & df["payment_type"].isin(valid_payment_types)
    & (df["PULocationID"].between(1, 263))
    & (df["DOLocationID"].between(1, 263))]



#trip distance in minutes
df["trip_duration_minutes"] = (df["tpep_dropoff_datetime"] -df["tpep_pickup_datetime"]).dt.total_seconds()/60

#average speed mph
df["speed_mph"] = df["trip_distance"] / (df["trip_duration_minutes"]/60)

#parse data with duraction 1-300 minutes and speeds from 1-80mph
df = df[(df["trip_duration_minutes"].between(1, 300)) &(df["speed_mph"].between(1, 80))]

/tmp/ipykernel_400/3226949441.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("nyc_taxi_data.csv", parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"])


In [ ]:
#join parsed data with lookup zones


df_lookup = pd.read_csv("taxi_zone_lookup.csv")

df_lookup_pu = df_lookup[["LocationID", "Borough", "Zone", "service_zone"]].rename(
    columns={"LocationID": "PULocationID", "Borough": "pu_borough", "Zone": "pu_zone", "service_zone": "pu_service_zone"})
df_lookup_do = df_lookup[["LocationID", "Borough", "Zone", "service_zone"]].rename(
    columns={"LocationID": "DOLocationID", "Borough": "do_borough", "Zone": "do_zone", "service_zone": "do_service_zone"})

df_merged = df.merge(df_lookup_pu, on='PULocationID', how='left')
df_merged = df_merged.merge(df_lookup_do, on='DOLocationID', how='left')


#batchload to database
engine = create_engine("sqlite:///nyc_taxi.db")
df_merged.to_sql(name="nyc_taxi_data", con=engine, if_exists="replace", index=False, chunksize=10000)

6051550

In [ ]:
#Extract Merge .csv file


df_merged.to_csv("nyc_taxi_merged.csv", index=False)

**SQL**

In [ ]:
import sqlite3



In [ ]:

conn = sqlite3.connect("nyc_taxi.db")
query = """ SELECT pu_borough, pu_zone, COUNT(*) AS trip_count FROM nyc_taxi_data GROUP BY pu_borough, pu_zone ORDER BY trip_count DESC LIMIT 20;"""
output = pd.read_sql(query, conn)
conn.close()
output

,pu_borough,pu_zone,trip_count
0,Manhattan,Upper East Side South,284022
1,Manhattan,Midtown Center,272979
2,Manhattan,Upper East Side North,264389
3,Manhattan,Midtown East,227958
4,Manhattan,Penn Station/Madison Sq West,220052
5,Manhattan,Times Sq/Theatre District,219931
6,Queens,JFK Airport,195642
7,Manhattan,Lincoln Square East,187467
8,Manhattan,Murray Hill,187384
9,Manhattan,Clinton East,186360


**Queries the most profitable routes from pickup to dropoff destination by total revenue and average fare of the ride

-routes must have at least 50 rides**

In [ ]:
print(df_merged.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'trip_duration_minutes', 'speed_mph', 'pu_borough', 'pu_zone', 'pu_service_zone', 'do_borough', 'do_zone', 'do_service_zone']


In [ ]:

conn = sqlite3.connect("nyc_taxi.db")
query = """SELECT
    PULocationID,
    pu_zone,
    pu_borough,
    DOLocationID,
    do_zone,
    do_borough,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance),2),
    ROUND(AVG(tip_amount),2),
    ROUND(AVG(fare_amount),2) AS avg_base_fare,
    ROUND(AVG(total_amount),2) AS avg_total_fare
FROM nyc_taxi_data
GROUP BY PULocationID, pu_zone, pu_borough, DOLocationID, do_zone, do_borough
HAVING COUNT(*) >= 50
ORDER BY avg_total_fare DESC
LIMIT 15;"""

output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query1.csv", index=False)
conn.close()
output

,PULocationID,pu_zone,pu_borough,DOLocationID,do_zone,do_borough,total_trips,"ROUND(AVG(trip_distance),2)","ROUND(AVG(tip_amount),2)",avg_base_fare,avg_total_fare
0,132,JFK Airport,Queens,1,Newark Airport,EWR,272,36.81,12.60,118.07,155.46
1,138,LaGuardia Airport,Queens,1,Newark Airport,EWR,68,29.21,13.13,100.15,134.86
2,262,Yorkville East,Manhattan,1,Newark Airport,EWR,51,22.36,15.00,80.00,110.28
3,151,Manhattan Valley,Manhattan,1,Newark Airport,EWR,72,21.56,14.66,77.35,107.53
4,236,Upper East Side North,Manhattan,1,Newark Airport,EWR,136,21.03,13.95,76.88,106.41
5,238,Upper West Side North,Manhattan,1,Newark Airport,EWR,145,20.42,13.76,74.52,103.66
6,141,Lenox Hill West,Manhattan,1,Newark Airport,EWR,64,20.08,12.14,75.57,103.60
7,163,Midtown North,Manhattan,1,Newark Airport,EWR,733,19.39,11.12,74.07,103.15
8,233,UN/Turtle Bay South,Manhattan,1,Newark Airport,EWR,188,19.21,12.04,74.03,103.08
9,237,Upper East Side South,Manhattan,1,Newark Airport,EWR,190,18.97,12.81,73.08,101.81


**Queries trip volume, average speed(congestination indicator), and average surcharge revenue**

-surcharge = mandatory extra fee(like $2.5 starting,ending or passing through manhattan south of 96th street) or tax(state/city fees) or rush hour/overnight
 surcharges or congestion surgecharge


In [ ]:

conn = sqlite3.connect("nyc_taxi.db")
query = """SELECT CASE CAST(strftime('%w', tpep_pickup_datetime) AS INTEGER)
  WHEN 0 THEN 'Sunday'
  WHEN 1 THEN 'Monday'
  WHEN 2 THEN 'Tuesday'
  WHEN 3 THEN 'Wednesday'
  WHEN 4 THEN 'Thursday'
  WHEN 5 THEN 'Friday'
  WHEN 6 THEN 'Saturday'
END AS day_of_week, CAST(strftime('%H', tpep_pickup_datetime) AS INTEGER) AS pickup_hour, COUNT(*) AS total_trips, ROUND(AVG(speed_mph), 2) AS avg_speed_mph,
ROUND(AVG(congestion_surcharge), 2) AS avg_congestion_surcharge, ROUND(AVG(total_amount), 2) AS avg_total_amount FROM nyc_taxi_data
GROUP BY strftime('%w', tpep_pickup_datetime), pickup_hour ORDER BY total_trips DESC;"""

output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query2.csv", index=False)
conn.close()
output

,day_of_week,pickup_hour,total_trips,avg_speed_mph,avg_congestion_surcharge,avg_total_amount
0,Thursday,18,75588,9.88,2.37,18.63
1,Friday,18,74302,10.06,2.36,18.52
2,Friday,19,70353,10.63,2.37,18.26
3,Thursday,19,68175,11.13,2.36,18.59
4,Wednesday,18,67690,10.22,2.37,18.38
...,...,...,...,...,...,...
163,Monday,4,3412,20.86,2.17,23.33
164,Tuesday,2,3366,17.99,2.22,18.69
165,Monday,3,2978,18.37,2.20,19.63
166,Tuesday,4,2828,20.89,2.15,23.19


**Queries the top pickup zones by driver net revenue per minute of active trip time **

In [ ]:

conn = sqlite3.connect("nyc_taxi.db")
query = """SELECT
    pu_borough,
    pu_zone,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_mins,
    ROUND(AVG(total_amount - tolls_amount), 2) AS avg_net_fare,
    ROUND(AVG((total_amount - tolls_amount) / trip_duration_minutes), 2) AS revenue_per_minute,
    ROUND(AVG(((total_amount - tolls_amount) / trip_duration_minutes) * 60), 2) AS projected_hourly_rate
FROM nyc_taxi_data
GROUP BY pu_borough, pu_zone
HAVING COUNT(*) >= 100
ORDER BY revenue_per_minute DESC
LIMIT 15;"""

output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query3.csv", index=False)
conn.close()
output

,pu_borough,pu_zone,total_trips,avg_duration_mins,avg_net_fare,revenue_per_minute,projected_hourly_rate
0,Queens,South Ozone Park,356,21.57,50.38,6.96,417.59
1,Manhattan,Randalls Island,338,15.92,56.60,4.62,277.31
2,Queens,Maspeth,641,17.72,55.76,3.60,216.22
3,Queens,Richmond Hill,210,19.13,39.90,3.50,210.07
4,Queens,Corona,644,22.10,60.60,3.07,184.23
5,Queens,Flushing Meadows-Corona Park,2996,24.23,63.91,3.03,181.56
6,Queens,Kew Gardens,526,21.62,49.67,2.82,169.27
7,Queens,Long Island City/Hunters Point,4505,15.03,32.22,2.62,157.26
8,Queens,Briarwood/Jamaica Hills,995,26.83,60.10,2.43,145.53
9,Queens,Rego Park,641,19.59,42.83,2.34,140.28


**Queries boroughs with more dropoffs than pickups

-where drivers may end up after finishing a ride where there is not much demand for a taxi
-passenger demand is not met**

In [ ]:
conn = sqlite3.connect("nyc_taxi.db")
query = """WITH pickups AS (
    SELECT pu_borough AS borough, COUNT(*) AS pickup_count
    FROM nyc_taxi_data
    WHERE pu_borough IS NOT NULL
    GROUP BY pu_borough
),
dropoffs AS (
    SELECT do_borough AS borough, COUNT(*) AS dropoff_count
    FROM nyc_taxi_data
    WHERE do_borough IS NOT NULL
    GROUP BY do_borough
)
SELECT
    p.borough,
    p.pickup_count,
    d.dropoff_count,
    (p.pickup_count - d.dropoff_count) AS net_trip_balance,
    ROUND((CAST(p.pickup_count AS FLOAT) / NULLIF(d.dropoff_count, 0)), 2) AS pickup_to_dropoff_ratio
FROM pickups p
JOIN dropoffs d ON p.borough = d.borough
ORDER BY net_trip_balance ASC;"""
output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query4.csv", index=False)
conn.close()
output

,borough,pickup_count,dropoff_count,net_trip_balance,pickup_to_dropoff_ratio
0,Brooklyn,47182,223925,-176743,0.21
1,Bronx,4286,32376,-28090,0.13
2,EWR,25,11077,-11052,0.00
3,Staten Island,246,1401,-1155,0.18
4,Queens,375188,291210,83978,1.29
5,Manhattan,5624623,5491561,133062,1.02


**Query passenger tipping rate by payment type and time of day

-check if differs between night and morning hours
-check for percentage that dont tip**

In [ ]:
conn = sqlite3.connect("nyc_taxi.db")

query = """
SELECT
    CASE
        WHEN CAST(strftime('%H', tpep_pickup_datetime) AS INTEGER) BETWEEN 6 AND 11 THEN 'Morning Rush (6-11AM)'
        WHEN CAST(strftime('%H', tpep_pickup_datetime) AS INTEGER) BETWEEN 12 AND 16 THEN 'Afternoon (12-4PM)'
        WHEN CAST(strftime('%H', tpep_pickup_datetime) AS INTEGER) BETWEEN 17 AND 21 THEN 'Evening Rush (5-9PM)'
        ELSE 'Night / Late Owl (10PM-5AM)'
    END AS time_window,
    COUNT(*) AS total_trips,
    ROUND(AVG(fare_amount), 2) AS avg_base_fare,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(CASE WHEN payment_type = 1 THEN (tip_amount / NULLIF(fare_amount, 0)) * 100 ELSE NULL END), 2) AS avg_cc_tip_pct,
    ROUND(100.0 * SUM(CASE WHEN tip_amount = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS zero_tip_pct
FROM nyc_taxi_data
GROUP BY time_window
ORDER BY avg_cc_tip_pct DESC;
"""
output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query5.csv", index=False)
conn.close()
output

,time_window,total_trips,avg_base_fare,avg_tip,avg_cc_tip_pct,zero_tip_pct
0,Evening Rush (5-9PM),1843952,12.04,2.28,26.89,24.63
1,Night / Late Owl (10PM-5AM),1028165,12.89,2.28,26.41,28.18
2,Afternoon (12-4PM),1687737,12.39,2.15,26.17,30.22
3,Morning Rush (6-11AM),1491696,11.79,2.08,25.39,26.69


**Query airport rides profitability vs standard city rides(rides in city) by yield per mile and toll expenses using service zones




service zone = 4 specific areas

-yellow zone = manhattan south of East 96th St & West 110th St (the core central business district).

-boro zone = Outer boroughs (Brooklyn, Queens, The Bronx, Staten Island) and Northern Manhattan (above 96th/110th St).

-airports = the 3 NY airports

-EWR = Newark Liberty International Airport

In [ ]:
conn = sqlite3.connect("nyc_taxi.db")
query = """
SELECT
    CASE
        WHEN pu_service_zone = 'Airports' OR do_service_zone = 'Airports' THEN 'Airport Route'
        ELSE 'Intra-City Route'
    END AS trip_type,
    COUNT(*) AS total_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_mins,
    ROUND(AVG(total_amount), 2) AS avg_total_fare,
    ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) AS revenue_per_mile,
    ROUND(AVG((total_amount - tolls_amount) / NULLIF(trip_duration_minutes, 0)), 2) AS driver_revenue_per_min
FROM nyc_taxi_data
WHERE pu_service_zone IS NOT NULL AND do_service_zone IS NOT NULL
GROUP BY trip_type;"""
output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query6.csv", index=False)
conn.close()
output

,trip_type,total_trips,avg_distance_miles,avg_duration_mins,avg_total_fare,revenue_per_mile,driver_revenue_per_min
0,Airport Route,416371,12.71,31.59,49.93,4.38,1.61
1,Intra-City Route,5635179,2.13,11.65,15.87,10.27,1.63


**Query the areas with the slowest speeds during rush hours(5-8 pm)

In [ ]:
conn = sqlite3.connect("nyc_taxi.db")
query = """
WITH corridor_speeds AS (
    SELECT
        pu_zone,
        do_zone,
        COUNT(*) AS trip_count,
        ROUND(AVG(speed_mph), 2) AS avg_speed_mph,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_mins,
        ROUND(AVG(trip_distance), 2) AS avg_distance_miles
    FROM nyc_taxi_data
    WHERE CAST(strftime('%H', tpep_pickup_datetime) AS INTEGER) BETWEEN 17 AND 20
    GROUP BY pu_zone, do_zone
    HAVING COUNT(*) >= 100
)
SELECT
    pu_zone,
    do_zone,
    trip_count,
    avg_speed_mph,
    avg_duration_mins,
    avg_distance_miles,
    DENSE_RANK() OVER (ORDER BY avg_speed_mph ASC) AS congestion_rank
FROM corridor_speeds
LIMIT 10;"""
output = pd.read_sql(query, conn)
with engine.connect() as conn:
    df_zone_summary = pd.read_sql(query, conn)

df_zone_summary.to_csv("nyc_query7.csv", index=False)
conn.close()
output

,pu_zone,do_zone,trip_count,avg_speed_mph,avg_duration_mins,avg_distance_miles,congestion_rank
0,Midtown East,Times Sq/Theatre District,2686,5.72,11.22,0.98,1
1,Midtown Center,Times Sq/Theatre District,3129,5.74,9.26,0.79,2
2,UN/Turtle Bay South,Times Sq/Theatre District,686,5.84,12.96,1.17,3
3,Penn Station/Madison Sq West,Midtown South,1088,5.90,8.53,0.76,4
4,Kips Bay,Penn Station/Madison Sq West,741,6.19,11.27,1.11,5
5,Midtown East,Midtown Center,1936,6.22,7.43,0.70,6
6,Sutton Place/Turtle Bay North,Times Sq/Theatre District,741,6.22,13.00,1.26,6
7,Penn Station/Madison Sq West,Penn Station/Madison Sq West,430,6.23,7.28,0.72,7
8,Times Sq/Theatre District,Times Sq/Theatre District,2132,6.25,6.89,0.64,8
9,UN/Turtle Bay South,Garment District,388,6.28,12.87,1.25,9
